# 01 - Data Understanding

**Project:** Student Placement & Salary Prediction  
**Goal of this notebook:** dataset ko samajhna: size, columns, data types, missing values, duplicates aur target variables.

**Notebook order:** 01 Data Understanding -> 02 EDA -> 03 Data Preprocessing -> 04 Feature Engineering -> 05 Model Training -> 06 Model Evaluation. Ye notebook `notebook/` folder ke andar se run karo (paths `../datasets` aur `../models` relative hain).

## 1. Libraries import karna
Sabse pehle data load aur display ke liye `pandas` aur `numpy` import karte hain.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)   # saare columns dikhane ke liye

## 2. Dataset load karna
`datasets/studentplacement.csv` ko read karte hain. Ye original raw file hai, ise kahin badla nahi gaya.

In [ ]:
DATA_PATH = '../datasets/studentplacement.csv'
raw = pd.read_csv(DATA_PATH)
print('Rows, Columns =', raw.shape)

## 3. Pehli 5 rows dekhna
Taaki pata chale har column me kis tarah ki values hain.

In [ ]:
raw.head()

## 4. Columns ka data dictionary
Har column ka matlab (column ke naam se samjha gaya) aur uska data type. Dataset ke saath koi official data dictionary nahi mili, isliye meaning naam se infer ki gayi hai.

| Group | Columns |
|---|---|
| Identity / personal | `student_id`, `age`, `gender` |
| Academic | `cgpa`, `branch`, `college_tier`, `backlogs`, `attendance_percentage` |
| Experience | `internships_count`, `projects_count`, `certifications_count`, `hackathons_participated`, `github_repos` |
| Skill scores | `coding_skill_score`, `aptitude_score`, `communication_skill_score`, `logical_reasoning_score`, `mock_interview_score` |
| Personality / lifestyle | `extracurricular_score`, `leadership_score`, `volunteer_experience`, `linkedin_connections`, `sleep_hours`, `study_hours_per_day` |
| Targets | `placement_status` (Placed / Not Placed), `salary_package_lpa` (salary in LPA) |

In [ ]:
pd.DataFrame({'dtype': raw.dtypes.astype(str), 'non_null': raw.notnull().sum(), 'unique_values': raw.nunique()})

## 5. Numerical columns ka statistical summary
`describe()` se min, max, mean, quartiles milte hain. Isse range aur outliers ka pehla andaza lagta hai (jaise scores 20-100, cgpa 4.5-10).

In [ ]:
raw.describe().T.round(2)

## 6. Categorical columns ki values
Text wale columns me kaun si categories hain aur kitne students har category me hain. Inhe preprocessing me numbers me convert karna padega.

In [ ]:
cat_cols = raw.select_dtypes(include=['object', 'string']).columns.tolist()
print('Categorical columns:', cat_cols)
for c in cat_cols:
    print('\n' + c)
    print(raw[c].value_counts().to_string())

## 7. Missing values aur duplicate rows
Data quality check: kya koi column khali hai ya koi row repeat hui hai?

In [ ]:
print('Total missing values :', int(raw.isnull().sum().sum()))
print('Duplicate rows         :', int(raw.duplicated().sum()))
print('Duplicate student_id    :', int(raw['student_id'].duplicated().sum()))

## 8. Target variables samajhna
Project me 2 targets hain: `placement_status` (class) aur `salary_package_lpa` (number). Dekhte hain ki Not Placed students ka salary kya hai, kyunki isse decide hoga ki salary model kin rows par train ho.

In [ ]:
print(raw['placement_status'].value_counts().to_string())
print('\nPlacement rate: {:.2f}%'.format((raw['placement_status'] == 'Placed').mean() * 100))
raw.groupby('placement_status')['salary_package_lpa'].describe().round(2)

## 9. Summary: data se kya samajh aaya

- Dataset me **100,000 rows** aur **26 columns** hain; **koi missing value nahi** aur **koi duplicate row nahi** hai.
- 4 text columns hain: `gender`, `branch`, `college_tier`, `volunteer_experience` (aur target `placement_status`).
- **Placed: 54,459 (54.46%)**, Not Placed: 45,541.
- **Not Placed students ka salary hamesha 0 hai**; Placed students ka salary 7.11 se 20.44 LPA tak hai. Isliye salary model sirf **Placed** students par train karna sahi hai (0 wali rows model ko bigaad deti).
- Is project ka main task: **Placed student ke features dekhkar uska salary package predict karna (regression)**. `placement_status` dashboard me sirf overview ke liye use hota hai.
- Aage 02_eda.ipynb me dekhenge ki kaun se features salary se related hain.